# GBE dataset: preliminary feasibility exploration

Here I inspect the data structure, including metadata, reconstruct derived reward variables, and check the ordering of happiness ratings.

## 1. Load the MATLAB data

In [ ]:
import numpy as np
import pandas as pd
from gbe_loader import load_gbe

In [2]:
participants = load_gbe()

## 2. People, trial data and reward terms

In [3]:
first_participant = participants[0]

print("Number of people:", len(participants))
print("Participant fields:", list(first_participant.keys()))
print("First participant ID:", first_participant["id"])
print("Available plays:", first_participant["nPlays"])
print("Trial data shape:", first_participant["data"].shape)
print("Started attempt:", first_participant["timesPlayed"])

Number of people: 47067
Participant fields: ['id', 'age', 'isFemale', 'location', 'lifeSatisfaction', 'education', 'nativeLanguage', 'deviceType', 'nPlays', 'timesPlayed', 'dayNumber', 'timeOfDay', 'designVersion', 'dataHdr', 'data']
First participant ID: 1
Available plays: 1
Trial data shape: (30, 15)
Started attempt: 4


This participant’s available play contains 30 trial rows and 15 columns. The column labels are stored separately in `dataHdr`.

In [4]:
headers = first_participant["dataHdr"]

for index, name in enumerate(headers):
    print(f"{index}: {name}")

0: nTrial
1: riskySide
2: certainValue
3: winValue
4: loseValue
5: empty
6: choseRisky
7: outcome
8: choiceRT
9: happiness
10: startValue
11: happinessRT
12: empty
13: spinDuration
14: spinAngle


In [5]:
task_data = first_participant["data"]

for index in [5, 12]:
    print(f"Column {index} ({headers[index]}), all NaN:",
          np.isnan(task_data[:, index]).all())

Column 5 (empty), all NaN: True
Column 12 (empty), all NaN: True


Two columns are labelled `empty`; both contain only missing values in this example play.

In [6]:
# columns needed to inspect choices, rewards and happiness
columns = [0, 2, 3, 4, 6, 7, 9]

example_trials = pd.DataFrame(
    task_data[:, columns],
    columns=headers[columns],
)

`choseRisky` records whether the participant chose the gamble (1) or the certain reward (0).

For the 50/50 gambles, expected value (EV) is the average of `winValue` and `loseValue`. On chosen gambles, reward prediction error (RPE) is the actual outcome minus this EV.

The reconstructed terms below follow the original model’s convention: `chosen_ev` is the gamble EV term and is zero on certain choices. Certain rewards are represented separately by `certain_reward`, and the gamble RPE term is also zero on certain choices.

In [7]:
offered_ev = (task_data[:, 3] + task_data[:, 4]) / 2
chose_gamble = task_data[:, 6] == 1

certain_reward = np.where(chose_gamble, 0, task_data[:, 2])
chosen_ev = np.where(chose_gamble, offered_ev, 0)
rpe = np.where(chose_gamble, task_data[:, 7] - offered_ev, 0)

# add reconstructed terms to the inspection table
example_trials["certain_reward"] = certain_reward
example_trials["chosen_ev"] = chosen_ev
example_trials["rpe"] = rpe

example_trials.head(6).style.format(
    {"chosen_ev": "{:.1f}", "rpe": "{:.1f}"},
    precision=0, na_rep="—"
).hide(axis="index")

nTrial,certainValue,winValue,loseValue,choseRisky,outcome,happiness,certain_reward,chosen_ev,rpe
1,-55,0,-220,1,-220,70,0,-110.0,-110.0
2,45,122,0,0,45,—,45,0.0,0.0
3,55,132,0,1,132,82,0,66.0,66.0
4,-35,0,-70,0,-35,—,-35,0.0,0.0
5,-55,0,-120,0,-55,—,-55,0.0,0.0
6,0,55,-11,1,55,78,0,22.0,33.0


In [8]:
# trial 1: EV = -110; RPE = -220 - (-110) = -110
# trial 2: certain reward = 45; gamble EV and RPE terms = 0
# trial 3: EV = 66; RPE = 132 - 66 = 66

np.testing.assert_allclose(certain_reward[:3], [0, 45, 0])
np.testing.assert_allclose(chosen_ev[:3], [-110, 0, 66])
np.testing.assert_allclose(rpe[:3], [-110, 0, 66])

print("First three trials match the manual calculations.")

First three trials match the manual calculations.


A zero outcome can still be better or worse than expected. These two trials from participant 1 check both cases against manual calculations.

In [9]:
# zero outcome, opposite RPE signs
# trial 8:  EV = -113/2 = -56.5; RPE = 0 - (-56.5) = +56.5
# trial 12: EV = 132/2 = 66;     RPE = 0 - 66 = -66
np.testing.assert_allclose(chosen_ev[[7, 11]], [-56.5, 66])
np.testing.assert_allclose(rpe[[7, 11]], [56.5, -66])

example_trials.loc[
    example_trials["nTrial"].isin([8, 12]),
    ["nTrial", "winValue", "loseValue", "choseRisky", "outcome", "chosen_ev", "rpe"],
].style.format(
    {"chosen_ev": "{:.1f}", "rpe": "{:.1f}"}, precision=0
).hide(axis="index")

nTrial,winValue,loseValue,choseRisky,outcome,chosen_ev,rpe
8,0,-113,1,0,-56.5,56.5
12,132,0,1,0,66.0,-66.0


## 3. Happiness ratings and intervening outcomes

According to the data dictionary, the rating stored in the first row was collected **before the task**. Later ratings were collected after the outcome on the corresponding trial. The table distinguishes the storage row from the number of trials completed at the rating.

In [10]:
happiness = task_data[:, 9]
has_rating = ~np.isnan(happiness)

rating_rows = task_data[has_rating, 0].astype(int)
rating_boundaries = rating_rows.copy()
rating_boundaries[0] = 0  # pre-task rating is stored in trial row 1

ratings = pd.DataFrame({
    "stored_trial_row": rating_rows,
    "completed_trials": rating_boundaries,
    "happiness": happiness[has_rating],
})

ratings.style.format(precision=0).hide(axis="index")

stored_trial_row,completed_trials,happiness
1,0,70
3,3,82
6,6,78
9,9,71
11,11,50
14,14,23
17,17,50
20,20,50
22,22,27
25,25,50


In [11]:
print("Trials between ratings:", np.diff(rating_boundaries))

Trials between ratings: [3 3 3 2 3 3 3 2 3 3 2]


Baseline happiness was measured before any trials. The interval between baseline and the second rating contains trials 1–3; the next interval contains trials 4–6, and so on. In this play, successive ratings are separated by two or three reward outcomes.

Here I look at the interval between the ratings after trials 9 and 11. Trials 10 and 11 are between these ratings.

In [12]:
left, right = rating_boundaries[3:5]

print(f"Starting happiness: {ratings["happiness"].iloc[3]:g}")
print(f"Ending happiness: {ratings["happiness"].iloc[4]:g}")

# Row positions 9 and 10 correspond to trials 10 and 11.
example_trials.iloc[left:right][
    ["nTrial", "choseRisky", "outcome", "certain_reward", "chosen_ev", "rpe"]
].style.format(
    precision=0, na_rep="—"
).hide(axis="index")

Starting happiness: 71
Ending happiness: 50


nTrial,choseRisky,outcome,certain_reward,chosen_ev,rpe
10,0,45,45,0,0
11,0,-55,-55,0,0


In this interval, the participant chose the certain option on both trials, receiving +45 and −55. Both gamble RPE terms are therefore zero, but the interval still contains reward and loss outcomes. Happiness decreased from 71 to 50.

## 4. Repeated plays and play metadata

Here I inspect a participant with multiple plays to see how these are stored.

In [13]:
for participant in participants:
    if participant["nPlays"] > 1:
        repeat_participant = participant
        break

print("Participant ID:", repeat_participant["id"])
print("Available plays:", repeat_participant["nPlays"])
print("Outer data shape:", repeat_participant["data"].shape)

repeat_plays = repeat_participant["data"]

Participant ID: 2
Available plays: 4
Outer data shape: (4,)


For participant 2, `data` contains four play matrices rather than a single trial matrix. The table below pairs each matrix with its play metadata.

In [14]:
play_summary = pd.DataFrame({
    "available_play": np.arange(1, len(repeat_plays) + 1),
    "timesPlayed": repeat_participant["timesPlayed"],
    "dayNumber": repeat_participant["dayNumber"],
    "timeOfDay": repeat_participant["timeOfDay"],
    "designVersion": repeat_participant["designVersion"],
    "trial_rows": [play.shape[0] for play in repeat_plays],
    "first_trial": [play[0, 0] for play in repeat_plays],
    "last_trial": [play[-1, 0] for play in repeat_plays],
})

play_summary.style.format({
    "timeOfDay": "{:.6f}",
    "first_trial": "{:.0f}",
    "last_trial": "{:.0f}",
}).hide(axis="index")

available_play,timesPlayed,dayNumber,timeOfDay,designVersion,trial_rows,first_trial,last_trial
1,1,0,0.710625,1,30,1,30
2,2,0,0.773542,1,30,1,30
3,3,0,0.775752,1,30,1,30
4,4,0,0.777986,1,30,1,30


Each available play contains 30 trials, with trial numbering restarting at 1. The data dictionary states that only completed plays with an internet connection at completion are included.

`available_play` numbers the plays stored in this file, whereas `timesPlayed` counts attempts started. These match for participant 2, but need not match for everyone: participant 1 has one available play recorded as attempt 4.

According to the dictionary, `dayNumber` counts days since the participant first played the task; 0 denotes that first day. `timeOfDay` records play completion as a fraction of a day in GMT (0.5 is noon). It is not a timestamp for each trial. `designVersion` identifies the task version.

All four plays shown here occurred on day 0 and used version 1. A trial needs a participant ID, an available-play index and a trial number to identify its position in the released data.

## 5. Conclusion

The inspected play shows how reward outcomes fall between successive happiness ratings. Several outcomes occur between ratings, so a rating interval is not a single RPE event.

The reward calculations match manual checks for the selected trials. Dataset-wide checks of trial structure, rating schedules, missing values and play metadata are still needed, alongside a decision about how to represent multiple outcomes between ratings.